# Data Cleaning

In this notebook, we will select a small set of useful features and prepare the data for linear regression.

The cleaned dataset will be saved as `data/cleaned_house_data.csv`.

In [1]:
from pathlib import Path

import pandas as pd

DATA_URL = "https://raw.githubusercontent.com/ageron/handson-ml/master/datasets/housing/housing.csv"

# Make paths work whether the notebook is opened from the project root or notebooks folder.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
CLEANED_DATA_PATH = DATA_DIR / "cleaned_house_data.csv"

In [2]:
housing_data = pd.read_csv(DATA_URL)
housing_data.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY


## Select useful features

We will keep a small number of columns so the project stays easy to understand.

In [3]:
selected_columns = [
    "total_rooms",
    "total_bedrooms",
    "housing_median_age",
    "households",
    "median_income",
    "ocean_proximity",
    "median_house_value",
]

house_data = housing_data[selected_columns].copy()
house_data.head()

,total_rooms,total_bedrooms,housing_median_age,households,median_income,ocean_proximity,median_house_value
0,880.0,129.0,41.0,126.0,8.3252,NEAR BAY,452600.0
1,7099.0,1106.0,21.0,1138.0,8.3014,NEAR BAY,358500.0
2,1467.0,190.0,52.0,177.0,7.2574,NEAR BAY,352100.0
3,1274.0,235.0,52.0,219.0,5.6431,NEAR BAY,341300.0
4,1627.0,280.0,52.0,259.0,3.8462,NEAR BAY,342200.0


## Check missing values

We check missing values before making any changes.

In [4]:
house_data.isna().sum()

total_rooms             0
total_bedrooms        207
housing_median_age      0
households              0
median_income           0
ocean_proximity         0
median_house_value      0
dtype: int64

## Handle missing values

`total_bedrooms` has missing values in this dataset. We will fill them with the median value because the median is simple and is less affected by very large values.

In [5]:
median_bedrooms = house_data["total_bedrooms"].median()
house_data["total_bedrooms"] = house_data["total_bedrooms"].fillna(median_bedrooms)

house_data.isna().sum()

total_rooms           0
total_bedrooms        0
housing_median_age    0
households            0
median_income         0
ocean_proximity       0
median_house_value    0
dtype: int64

## Remove duplicate rows

Duplicate rows can accidentally give repeated examples to the model. We will remove them if any exist.

In [6]:
duplicate_count = house_data.duplicated().sum()
print(f"Duplicate rows before cleaning: {duplicate_count}")

house_data = house_data.drop_duplicates()

print(f"Rows after removing duplicates: {len(house_data)}")

Duplicate rows before cleaning: 0
Rows after removing duplicates: 20640


## Encode the location category

Linear regression needs numeric input. `ocean_proximity` is text, so we convert it into simple 0/1 columns using one-hot encoding.

In [7]:
house_data_encoded = pd.get_dummies(house_data, columns=["ocean_proximity"], drop_first=True)
house_data_encoded.head()

,total_rooms,total_bedrooms,housing_median_age,households,median_income,median_house_value,ocean_proximity_INLAND,ocean_proximity_ISLAND,ocean_proximity_NEAR BAY,ocean_proximity_NEAR OCEAN
0,880.0,129.0,41.0,126.0,8.3252,452600.0,False,False,True,False
1,7099.0,1106.0,21.0,1138.0,8.3014,358500.0,False,False,True,False
2,1467.0,190.0,52.0,177.0,7.2574,352100.0,False,False,True,False
3,1274.0,235.0,52.0,219.0,5.6431,341300.0,False,False,True,False
4,1627.0,280.0,52.0,259.0,3.8462,342200.0,False,False,True,False


## Check the cleaned dataset

In [8]:
house_data_encoded.info()

<class 'pandas.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 10 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   total_rooms                 20640 non-null  float64
 1   total_bedrooms              20640 non-null  float64
 2   housing_median_age          20640 non-null  float64
 3   households                  20640 non-null  float64
 4   median_income               20640 non-null  float64
 5   median_house_value          20640 non-null  float64
 6   ocean_proximity_INLAND      20640 non-null  bool   
 7   ocean_proximity_ISLAND      20640 non-null  bool   
 8   ocean_proximity_NEAR BAY    20640 non-null  bool   
 9   ocean_proximity_NEAR OCEAN  20640 non-null  bool   
dtypes: bool(4), float64(6)
memory usage: 1.0 MB


In [9]:
house_data_encoded.describe()

,total_rooms,total_bedrooms,housing_median_age,households,median_income,median_house_value
count,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000
mean,2635.763081,536.838857,28.639486,499.539680,3.870671,206855.816909
std,2181.615252,419.391878,12.585558,382.329753,1.899822,115395.615874
min,2.000000,1.000000,1.000000,1.000000,0.499900,14999.000000
25%,1447.750000,297.000000,18.000000,280.000000,2.563400,119600.000000
50%,2127.000000,435.000000,29.000000,409.000000,3.534800,179700.000000
75%,3148.000000,643.250000,37.000000,605.000000,4.743250,264725.000000
max,39320.000000,6445.000000,52.000000,6082.000000,15.000100,500001.000000


## Save the cleaned data

In [10]:
DATA_DIR.mkdir(parents=True, exist_ok=True)
house_data_encoded.to_csv(CLEANED_DATA_PATH, index=False)

print(f"Cleaned data saved to: {CLEANED_DATA_PATH}")

Cleaned data saved to: C:\Users\milli\OneDrive\Documents\New project\house-price-prediction\data\cleaned_house_data.csv


## Cleaning summary

Write your notes here after running the notebook. The main decisions were:

- Kept a small set of understandable features.
- Filled missing `total_bedrooms` values with the median.
- Removed duplicate rows if any were present.
- Converted `ocean_proximity` into numeric columns so the model can use it.
- Saved the cleaned dataset for the model training notebook.